# Master Verification: Planetary Polygons Paper

This notebook independently verifies every key numerical constant claimed in the paper
"Planetary Polar Polygons as Constrained Energy Minima of Point Vortex Rings."

Each section imports from the computational modules in `src/`, recomputes the claimed
value, asserts correctness, and prints a PASS/FAIL line. A final summary cell tallies
all results.

**Verified claims:**
1. Havelock eigenvalues $\lambda_m = (N{-}1) - m(N{-}m)/2$ for $N=3\ldots8$
2. N=7 quartic bifurcation: unconstrained $153/7$, constrained $135/7$, $\alpha_0 = 45/14$
3. S^2 rational thresholds: $\{1/3, 1/5, 1/7, 1/19\}$ for $N=3,4,5,6$
4. H^2 threshold: $\xi^* = 8 - 3\sqrt{7} \approx 0.0627$, norm identity $(8-3\sqrt7)(8+3\sqrt7)=1$
5. Blob correction table: $c_m$ values for $N=3\ldots8$
6. Cat's-eye braid fraction scaling: $O(1-\eta)$ with exponent $\approx 1.08$

In [ ]:
# ============================================================
# Setup: paths, imports, pass/fail tracking
# ============================================================
import sys
import os

# Add src/ to the Python path
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
SRC_DIR = os.path.join(REPO_ROOT, 'src')
sys.path.insert(0, SRC_DIR)

import numpy as np
from fractions import Fraction

# Global pass/fail counters
_pass_count = 0
_fail_count = 0
_results = []

def check(name, condition, detail=''):
    """Record and print a PASS/FAIL check."""
    global _pass_count, _fail_count
    if condition:
        _pass_count += 1
        tag = 'PASS'
    else:
        _fail_count += 1
        tag = 'FAIL'
    msg = f'[{tag}] {name}'
    if detail:
        msg += f'  ({detail})'
    print(msg)
    _results.append((tag, name, detail))

print(f'Repository root: {REPO_ROOT}')
print(f'Source dir:      {SRC_DIR}')
print(f'numpy version:   {np.__version__}')
print('Setup complete.')

## 1. Run test suite (non-scipy tests)

In [ ]:
import subprocess

# Run all non-scipy test files
test_files = [
    'tests/test_algebraic_thresholds.py',
    'tests/test_h2_stability.py',
    'tests/test_riemannian_havelock.py',
    'tests/test_blob_correction.py',
    'tests/test_circulation_disorder.py',
    'tests/test_sign_rule.py',
    'tests/test_variational.py',
    'tests/test_n7_bifurcation.py',
    'tests/test_catseye_decomposition.py',
]

cmd = [
    sys.executable, '-m', 'pytest',
    '--tb=short', '-q',
] + test_files

result = subprocess.run(
    cmd,
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
    timeout=300,
    env={**os.environ, 'PYTHONPATH': SRC_DIR},
)

print('--- pytest stdout ---')
print(result.stdout)
if result.stderr:
    print('--- pytest stderr ---')
    print(result.stderr)
print(f'Return code: {result.returncode}')

check('pytest suite exits cleanly', result.returncode == 0,
      f'return code {result.returncode}')

## 2. Havelock eigenvalues

Paper claim (Havelock 1931, re-derived in Section 3):
$$\lambda_m = (N{-}1) - \frac{m(N{-}m)}{2}, \quad m = 0, 1, \ldots, N{-}1$$

Stability iff all $\lambda_m > 0$, giving $N \le 7$ on the flat plane.

In [ ]:
from planetary_polygons.extensions.riemannian_havelock import havelock_sum, havelock_exact

print('Havelock eigenvalue verification: lambda_m = (N-1) - m(N-m)/2')
print(f'{"N":>3} {"m":>3} {"lambda_m (formula)":>20} {"T_m (numerical sum)":>20} {"T_m (exact m(N-m))":>20}')
print('-' * 80)

all_havelock_ok = True
for N in range(3, 9):
    for m in range(0, N):
        # Paper formula
        lam_m = Fraction(N - 1) - Fraction(m * (N - m), 2)
        
        if m == 0:
            # T_0 = 0 by definition, lambda_0 = N-1
            T_exact = Fraction(0)
            T_numerical = 0.0
        else:
            T_exact = havelock_exact(N, m)  # = m(N-m)
            T_numerical = havelock_sum(N, m)
        
        # Verify T_m = m(N-m) numerically
        if m > 0:
            ok = abs(T_numerical - float(T_exact)) < 1e-10
            all_havelock_ok = all_havelock_ok and ok
        
        # Verify lambda_m formula
        lam_check = Fraction(N - 1) - T_exact / 2
        ok2 = (lam_m == lam_check)
        all_havelock_ok = all_havelock_ok and ok2
        
        print(f'{N:>3} {m:>3} {float(lam_m):>20.6f} {T_numerical:>20.6f} {str(T_exact):>20}')

check('Havelock eigenvalue formula lambda_m = (N-1) - m(N-m)/2', all_havelock_ok)

# Verify stability boundary: N<=7 stable, N=8 unstable
print('\nStability check (min lambda_m > 0 iff N <= 7):')
for N in range(3, 10):
    lam_min = min(Fraction(N - 1) - Fraction(m * (N - m), 2) for m in range(1, N))
    stable = lam_min >= 0
    status = 'STABLE' if stable else 'UNSTABLE'
    print(f'  N={N}: min lambda = {float(lam_min):.4f} -> {status}')

# N<=7 stable
for N in range(3, 8):
    lam_min = min(Fraction(N - 1) - Fraction(m * (N - m), 2) for m in range(1, N))
    check(f'N={N} flat-plane stable', lam_min >= 0, f'min lambda = {float(lam_min)}')

# N=8 unstable
lam_min_8 = min(Fraction(8 - 1) - Fraction(m * (8 - m), 2) for m in range(1, 8))
check('N=8 flat-plane unstable', lam_min_8 < 0, f'min lambda = {float(lam_min_8)}')

## 3. N=7 quartic bifurcation

Paper Section 6.3 claims:
- Unconstrained quartic: $d^4H/dt^4 = 153/7 \approx 21.857$
- Constrained quartic: $135/7 \approx 19.286$
- Constraint correction: $\Delta = 18/7 \approx 2.571$
- Leading amplitude coefficient: $\alpha_0 = 45/14 \approx 3.214$

In [ ]:
from planetary_polygons.extensions.n7_bifurcation import (
    quartic_exact_n7,
    constrained_quartic_exact,
    constraint_correction_exact,
    alpha_0_exact,
    unconstrained_quartic,
    constrained_quartic_n7,
)

# Exact rational values
q_unc = quartic_exact_n7()              # 153/7
q_con = constrained_quartic_exact()     # 135/7
delta = constraint_correction_exact()   # 18/7
a0 = alpha_0_exact()                    # 45/14

print('N=7 quartic bifurcation (exact rational):')
print(f'  Unconstrained quartic = {q_unc} = {float(q_unc):.6f}')
print(f'  Constrained quartic   = {q_con} = {float(q_con):.6f}')
print(f'  Correction Delta      = {delta} = {float(delta):.6f}')
print(f'  alpha_0               = {a0} = {float(a0):.6f}')

check('Unconstrained quartic = 153/7', q_unc == Fraction(153, 7), str(q_unc))
check('Constrained quartic = 135/7', q_con == Fraction(135, 7), str(q_con))
check('Correction 153/7 - 135/7 = 18/7', delta == Fraction(18, 7), str(delta))
check('alpha_0 = 45/14', a0 == Fraction(45, 14), str(a0))

# Cross-check: numerical unconstrained quartic via analytic per-pair formula
q_num_m3 = unconstrained_quartic(7, 3)
q_num_m4 = unconstrained_quartic(7, 4)
print(f'\nNumerical unconstrained quartic (m=3): {q_num_m3:.6f}')
print(f'Numerical unconstrained quartic (m=4): {q_num_m4:.6f}')
print(f'Exact 153/7 = {float(Fraction(153, 7)):.6f}')

check('Numerical unconstrained quartic m=3 matches 153/7',
      abs(q_num_m3 - float(Fraction(153, 7))) < 1e-6,
      f'err = {abs(q_num_m3 - float(Fraction(153, 7))):.2e}')
check('Z_7 symmetry: m=3 and m=4 give same quartic',
      abs(q_num_m3 - q_num_m4) < 1e-6,
      f'diff = {abs(q_num_m3 - q_num_m4):.2e}')

# Numerical constrained quartic via Richardson extrapolation
q_con_num = constrained_quartic_n7()
print(f'\nNumerical constrained quartic (Richardson): {q_con_num:.4f}')
print(f'Exact 135/7 = {float(Fraction(135, 7)):.4f}')
check('Numerical constrained quartic matches 135/7 to 1%',
      abs(q_con_num - float(Fraction(135, 7))) / float(Fraction(135, 7)) < 0.01,
      f'value = {q_con_num:.6f}, exact = {float(Fraction(135, 7)):.6f}')

## 4. S^2 exact rational thresholds

Paper Section 4.2: On the sphere, the N-gon loses stability at
$$\xi_{\mathrm{crit}} = \frac{(N{-}1) - m(N{-}m)/2}{(N{-}1) + m(N{-}m)/2}, \quad m = \lfloor N/2 \rfloor$$

| N | $\xi_{\mathrm{crit}}$ |
|---|---|
| 3 | 1/3 |
| 4 | 1/5 |
| 5 | 1/7 |
| 6 | 1/19 |
| $\ge 7$ | None (already unstable) |

In [ ]:
from planetary_polygons.extensions.algebraic_thresholds import sphere_stability_threshold

expected_s2 = {
    3: Fraction(1, 3),
    4: Fraction(1, 5),
    5: Fraction(1, 7),
    6: Fraction(1, 19),
}

print('S^2 destabilization thresholds (exact rational):')
for N in range(3, 9):
    xi_crit = sphere_stability_threshold(N)
    if N in expected_s2:
        print(f'  N={N}: xi_crit = {xi_crit} (expected {expected_s2[N]})')
        check(f'S^2 threshold N={N} = {expected_s2[N]}',
              xi_crit == expected_s2[N],
              f'got {xi_crit}')
    else:
        print(f'  N={N}: xi_crit = {xi_crit} (expected None)')
        check(f'S^2 threshold N={N} is None (unstable in flat limit)',
              xi_crit is None,
              f'got {xi_crit}')

## 5. H^2 threshold: $\xi^* = 8 - 3\sqrt{7}$

Paper Section 4.3: The 7-to-8 stability transition on H^2 occurs at
$$\xi^* = 8 - 3\sqrt{7} \approx 0.0627$$

This is the inverse fundamental unit $\varepsilon^{-1}$ of $\mathbb{Z}[\sqrt{7}]$:
$$\varepsilon = 8 + 3\sqrt{7}, \quad \varepsilon \cdot \varepsilon^{-1} = (8+3\sqrt7)(8-3\sqrt7) = 64 - 63 = 1$$

In [ ]:
from planetary_polygons.extensions.h2_stability import XI_STAR_78, GAMMA_78, C1_h2_exact
from planetary_polygons.extensions.algebraic_thresholds import h2_stability_threshold

xi_star = 8 - 3 * np.sqrt(7)
eps_fund = 8 + 3 * np.sqrt(7)  # fundamental unit
norm_product = xi_star * eps_fund

print(f'xi* = 8 - 3*sqrt(7) = {xi_star:.10f}')
print(f'eps  = 8 + 3*sqrt(7) = {eps_fund:.10f}')
print(f'xi* * eps = {norm_product:.15f} (should be 1.0)')
print(f'64 - 63 = {64 - 9*7}')

check('xi* approx 0.0627', abs(xi_star - 0.0627) < 0.001,
      f'xi* = {xi_star:.6f}')
check('(8-3*sqrt7)(8+3*sqrt7) = 1 (norm identity)',
      abs(norm_product - 1.0) < 1e-12,
      f'product = {norm_product}')
check('XI_STAR_78 module constant matches',
      abs(XI_STAR_78 - xi_star) < 1e-14,
      f'module value = {XI_STAR_78}')

# Verify: C1(H2, xi*) should equal the critical value m(N-m)/2 for N=8, m=4
# m(N-m)/2 = 4*4/2 = 8
C1_at_threshold = C1_h2_exact(8, xi_star)
print(f'\nC1(H2, xi*) at N=8 = {C1_at_threshold:.10f} (should be 8.0)')
check('C1(H2, xi*) = 8 at N=8 threshold',
      abs(C1_at_threshold - 8.0) < 1e-8,
      f'C1 = {C1_at_threshold:.10f}')

# Verify through algebraic_thresholds module
xi_alg, D_alg, field_alg = h2_stability_threshold(8)
print(f'\nAlgebraic threshold module: xi*(8) = {xi_alg:.10f}, field = {field_alg}')
check('h2_stability_threshold(8) matches xi* = 8-3*sqrt(7)',
      abs(xi_alg - xi_star) < 1e-8,
      f'xi_alg = {xi_alg:.10f}')
check('H2 N=8 field extension is Q(sqrt(7))',
      field_alg == 'Q(sqrt(7))',
      f'field = {field_alg}')

# GAMMA_78 = 1/xi*^2 = 127 + 48*sqrt(7)
gamma_computed = 1.0 / xi_star**2
gamma_exact = 127 + 48 * np.sqrt(7)
print(f'\ngamma = 1/xi*^2 = {gamma_computed:.6f}')
print(f'127 + 48*sqrt(7) = {gamma_exact:.6f}')
check('gamma = 1/xi*^2 = 127+48*sqrt(7)',
      abs(gamma_computed - gamma_exact) < 1e-6,
      f'diff = {abs(gamma_computed - gamma_exact):.2e}')

## 6. Blob correction table

Paper Section 6.4, Proposition 4.2:
$$\lambda_m(\varepsilon) = \lambda_m(0) + c_m \, \varepsilon^2 + O(\varepsilon^4)$$

| N | $c_m$ |
|---|---|
| 3 | $-1$ |
| 4 | $-1$ |
| 5 | $-1$ |
| 6 | $13/12$ |
| 7 | $4$ |
| 8 | $11$ |

$c_m < 0$ for $N \le 5$ (blob destabilises), $c_m > 0$ for $N \ge 6$ (blob stabilises).

In [ ]:
from planetary_polygons.extensions.blob_correction import blob_correction_table

expected_cm = {
    3: Fraction(-1),
    4: Fraction(-1),
    5: Fraction(-1),
    6: Fraction(13, 12),
    7: Fraction(4),
    8: Fraction(11),
}

table = blob_correction_table(range(3, 9))

print('Blob correction table:')
print(f'{"N":>3}  {"shift":>10}  {"P_m":>10}  {"c_m (computed)":>14}  {"c_m (expected)":>14}  {"match":>6}')
print('-' * 70)

for N in range(3, 9):
    entry = table[N]
    cm_computed = entry['cm']
    cm_expected = float(expected_cm[N])
    match = abs(cm_computed - cm_expected) < 0.01
    
    print(f'{N:>3}  {entry["shift"]:>10.4f}  {entry["Pm"]:>10.4f}  '
          f'{cm_computed:>14.6f}  {cm_expected:>14.6f}  {"OK" if match else "FAIL":>6}')
    
    check(f'Blob c_m for N={N} = {expected_cm[N]}',
          match,
          f'computed {cm_computed:.6f}, expected {cm_expected:.6f}')

# Verify sign pattern
print('\nSign pattern verification:')
for N in range(3, 9):
    cm = table[N]['cm']
    if N <= 5:
        check(f'c_m < 0 for N={N} (blob destabilises)',
              cm < 0, f'c_m = {cm:.4f}')
    else:
        check(f'c_m > 0 for N={N} (blob stabilises)',
              cm > 0, f'c_m = {cm:.4f}')

## 7. Cat's-eye braid fraction scaling

Paper Section 5.4.3: In the Stuart vortex concentration limit ($\varepsilon \to 1$),
the braid fraction of total circulation vanishes as $O(1-\varepsilon)$ with
exponent $\approx 1.08$. This establishes the conditions for Proposition 8
(inertia preservation) to apply, closing the Rossby bridge gap.

In [ ]:
from planetary_polygons.extensions.catseye_decomposition import (
    stuart_vorticity,
    stuart_stream_function,
    stuart_separatrix_level,
    stuart_background_vorticity,
    classify_regions,
    fit_scaling_law,
)

sigma = 0.5
ny, nx = 512, 512
domain_half = 6.0 * sigma
y_grid = np.linspace(-domain_half, domain_half, ny)
x_grid = np.linspace(0, 2 * np.pi, nx, endpoint=False)
dy = y_grid[1] - y_grid[0]
dx = x_grid[1] - x_grid[0]
dA = dy * dx

eps_values = [0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 0.98, 0.99]
one_minus = []
braid_fracs = []

print(f'{"eps":>8} {"1-eps":>8} {"braid_frac":>12}')
print('-' * 32)

for eps in eps_values:
    omega = stuart_vorticity(y_grid, x_grid, sigma=sigma, eps=eps, n=6)
    Psi = stuart_stream_function(y_grid, x_grid, sigma=sigma, eps=eps, n=6)
    psi_sep = stuart_separatrix_level(sigma, eps)
    mask = classify_regions(Psi, psi_sep, 0.05 * sigma**2)
    
    G_braid = np.sum(omega[mask == 2]) * dA
    G_total = np.sum(omega) * dA
    frac = G_braid / G_total
    
    one_minus.append(1.0 - eps)
    braid_fracs.append(frac)
    print(f'{eps:8.3f} {1-eps:8.4f} {frac:12.6f}')

# Fit scaling law
alpha, C, r2 = fit_scaling_law(one_minus, braid_fracs)
print(f'\nScaling fit: braid_frac ~ (1-eps)^{alpha:.3f}')
print(f'R^2 = {r2:.6f}')

check('Braid fraction exponent > 0.8 (vanishes as O(1-eps))',
      alpha > 0.8,
      f'exponent = {alpha:.3f}')
check('Braid fraction exponent < 1.4',
      alpha < 1.4,
      f'exponent = {alpha:.3f}')
check('Braid fraction exponent approx 1.08 (within 0.2)',
      abs(alpha - 1.08) < 0.2,
      f'exponent = {alpha:.3f}, target = 1.08')
check('Braid scaling fit R^2 > 0.99',
      r2 > 0.99,
      f'R^2 = {r2:.6f}')
check('Braid fraction at eps=0.99 < 1.5%',
      braid_fracs[-1] < 0.015,
      f'frac = {braid_fracs[-1]:.4f}')

## 8. Additional cross-checks

Supplementary verifications for completeness.

In [ ]:
# --- Riemannian Havelock identity: T_m = m(N-m) ---
from planetary_polygons.extensions.riemannian_havelock import havelock_sum

print('Riemannian Havelock identity: T_m = m(N-m)')
max_err = 0.0
for N in [4, 6, 7, 8, 10]:
    for m in range(1, N):
        T_num = havelock_sum(N, m)
        T_exact = m * (N - m)
        err = abs(T_num - T_exact)
        max_err = max(max_err, err)

print(f'  Max error across all (N,m) pairs: {max_err:.2e}')
check('Havelock sum T_m = m(N-m) to machine precision',
      max_err < 1e-10,
      f'max error = {max_err:.2e}')

# --- C1 flat limit: C1(H2, xi=0) = N-1, C1(S2, xi=0) = N-1 ---
from planetary_polygons.extensions.riemannian_havelock import C1_hyperbolic, C1_sphere

print('\nFlat-limit check: C1(surface, xi=0) = N-1')
for N in [4, 6, 8]:
    c1_h2 = C1_hyperbolic(N, 0.0)
    c1_s2 = C1_sphere(N, 0.0)
    print(f'  N={N}: C1_H2(0) = {c1_h2:.6f}, C1_S2(0) = {c1_s2:.6f}, N-1 = {N-1}')

check('C1(H2, 0) = N-1 for all N',
      all(abs(C1_hyperbolic(N, 0.0) - (N-1)) < 1e-12 for N in range(3, 12)),
      'tested N=3..11')
check('C1(S2, 0) = N-1 for all N',
      all(abs(C1_sphere(N, 0.0) - (N-1)) < 1e-12 for N in range(3, 12)),
      'tested N=3..11')

# --- H2 field extension pattern for even N ---
# Even N: Q(sqrt(squarefree(N-1))), Odd N: Q(sqrt(squarefree(N-3)))
# N=10: squarefree(9) = 1, so field is Q (rational threshold)
from planetary_polygons.extensions.algebraic_thresholds import h2_stability_threshold

print('\nH2 field extensions (N >= 8):')
expected_fields = {
    8: 'Q(sqrt(7))',
    9: 'Q(sqrt(6))',
    10: 'Q',  # squarefree(9) = 1
}
for N in [8, 9, 10]:
    xi_val, D, field = h2_stability_threshold(N)
    print(f'  N={N}: xi* = {xi_val:.6f}, field = {field}')
    if N in expected_fields:
        check(f'H2 field extension N={N} = {expected_fields[N]}',
              field == expected_fields[N],
              f'got {field}')

# N=10: rational threshold xi* = 1/7
xi_10, _, _ = h2_stability_threshold(10)
check('N=10 H2 threshold is rational: xi* = 1/7',
      abs(xi_10 - 1.0/7) < 1e-8,
      f'xi*(10) = {xi_10:.10f}, 1/7 = {1/7:.10f}')

## Final Summary

In [ ]:
print('=' * 70)
print('MASTER VERIFICATION SUMMARY')
print('=' * 70)
print(f'  Total checks: {_pass_count + _fail_count}')
print(f'  PASSED:       {_pass_count}')
print(f'  FAILED:       {_fail_count}')
print('=' * 70)

if _fail_count > 0:
    print('\nFailed checks:')
    for tag, name, detail in _results:
        if tag == 'FAIL':
            print(f'  [FAIL] {name}  ({detail})')
else:
    print('\nAll checks passed. Every numerical claim in the paper is verified.')

print('\n--- End of master verification ---')